In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import seaborn as sns #...
import math
myplate=sns.color_palette("Paired")
sns.set(style='white', palette=myplate)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import warnings
warnings.filterwarnings("ignore")

clrs = ["#ADD8E6","#FF0000"]
mypalette = sns.set_palette(sns.color_palette(clrs))

# EDA

In [ ]:
sal_path = "/kaggle/input/jobs-dataset-from-glassdoor/eda_data.csv"
sal = pd.read_csv(sal_path, index_col = 0)
sal.head()

In [ ]:
sal.describe()

In [ ]:
sal.shape

In [ ]:
sal.columns

In [ ]:
colonnes = ['job_simp', 'Rating', 'Company Name', 'Size', 'Founded', 'Industry', 'Sector', 'min_salary', 'avg_salary', 'max_salary','age', 'Type of ownership']  # Remplacez par vos noms de colonnes
df = sal[colonnes].copy()

In [ ]:
print(df)

In [ ]:
# 1. RENOMMER LES COLONNES
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("✓ Colonnes renommées")

# 2. NETTOYAGE
df['company_name'] = df['company_name'].str.replace(r'\n\d+\.\d+', '', regex=True)
df = df.replace('na', np.nan).replace(-1, np.nan)
print("✓ Données nettoyées")

# 3. IDENTIFIER LES COLONNES CATÉGORIELLES ET NUMÉRIQUES

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Colonnes catégorielles: {categorical_cols}")
print(f"Colonnes numériques: {numerical_cols}")

# 4. GÉRER LES VALEURS MANQUANTES
print(f"Avant: {df.isnull().sum().sum()} valeurs manquantes")

# Pour les colonnes numériques : remplir avec la médiane
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Pour les colonnes catégorielles : remplir avec 'unknown'
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('unknown', inplace=True)

print(f"Après: {df.isnull().sum().sum()} valeurs manquantes")

In [ ]:
df["rating"] = df["rating"].round().astype("Int64")
df["founded"] = df["founded"].round().astype("Int64")
df["avg_salary"] = df["avg_salary"].round().astype("Int64")
df["age"] = df["avg_salary"].round().astype("Int64")


## Analyse des données

In [ ]:
# 1. DISTRIBUTION DES SALAIRES PAR JOB_SIMP
print("\n1. Distribution des salaires par type de poste")
plt.figure(figsize=(14, 8))
df_sorted = df.groupby('job_simp')['avg_salary'].median().sort_values(ascending=False)
top_jobs = df_sorted.head(10).index

sns.boxplot(data=df[df['job_simp'].isin(top_jobs)], 
            x='avg_salary', 
            y='job_simp', 
            order=top_jobs,
            palette='viridis')
plt.title('Distribution des Salaires par Type de Poste (Top 10)', fontsize=16, fontweight='bold')
plt.xlabel('Salaire Moyen (k$)', fontsize=12)
plt.ylabel('Type de Poste', fontsize=12)
plt.tight_layout()
plt.savefig('salary_distribution_by_job.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: salary_distribution_by_job.png")
plt.show()

In [ ]:
# 2. MATRICE DE CORRÉLATION
print("\n2. Matrice de corrélation des variables numériques...")
plt.figure(figsize=(12, 10))
numeric_cols = ['rating', 'min_salary', 'avg_salary', 'max_salary', 'age']
correlation_matrix = df[numeric_cols].corr()

sns.heatmap(correlation_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation des Variables Numériques', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: correlation_matrix.png")
plt.show()

In [ ]:
# 3. RÉPARTITION PAR SECTEUR
print("\n3. Répartition des emplois par secteur...")
plt.figure(figsize=(14, 8))
sector_counts = df['sector'].value_counts().head(10)

plt.barh(range(len(sector_counts)), sector_counts.values, color='steelblue')
plt.yticks(range(len(sector_counts)), sector_counts.index)
plt.xlabel('Nombre d\'Offres', fontsize=12)
plt.ylabel('Secteur', fontsize=12)
plt.title('Top 10 des Secteurs avec le Plus d\'Offres d\'Emploi', fontsize=16, fontweight='bold')

for i, v in enumerate(sector_counts.values):
    plt.text(v + 1, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('jobs_by_sector.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: jobs_by_sector.png")
plt.show()

In [ ]:
# 4. RELATION ENTRE L'ÂGE DE L'ENTREPRISE ET LE SALAIRE
print("\n4. Relation entre l'âge de l'entreprise et le salaire...")
plt.figure(figsize=(12, 7))

# Créer des bins d'âge
df['age_bin'] = pd.cut(df['age'], bins=[0, 10, 20, 50, 100, 200], 
                        labels=['0-10 ans', '11-20 ans', '21-50 ans', '51-100 ans', '100+ ans'])

sns.boxplot(data=df, x='age_bin', y='avg_salary', palette='Set2')
plt.title('Salaire Moyen selon l\'Âge de l\'Entreprise', fontsize=16, fontweight='bold')
plt.xlabel('Âge de l\'Entreprise', fontsize=12)
plt.ylabel('Salaire Moyen (k$)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('salary_by_company_age.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: salary_by_company_age.png")
plt.show()

In [ ]:
# 5. DISTRIBUTION DES RATINGS
print("\n5. Distribution des ratings des entreprises...")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogramme
axes[0].hist(df['rating'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(df['rating'].mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {df["rating"].mean():.2f}')
axes[0].axvline(df['rating'].median(), color='green', linestyle='--', linewidth=2, label=f'Médiane: {df["rating"].median():.2f}')
axes[0].set_xlabel('Rating', fontsize=12)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].set_title('Distribution des Ratings', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot rating vs salary
axes[1].scatter(df['rating'], df['avg_salary'], alpha=0.5, s=50, color='coral')
axes[1].set_xlabel('Rating', fontsize=12)
axes[1].set_ylabel('Salaire Moyen (k$)', fontsize=12)
axes[1].set_title('Relation Rating vs Salaire', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Ajouter une ligne de tendance
z = np.polyfit(df['rating'].dropna(), df.loc[df['rating'].notna(), 'avg_salary'], 1)
p = np.poly1d(z)
axes[1].plot(df['rating'].sort_values(), p(df['rating'].sort_values()), 
             "r--", linewidth=2, label='Tendance')
axes[1].legend()

plt.tight_layout()
plt.savefig('rating_analysis.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: rating_analysis.png")
plt.show()


In [ ]:
# 6. SALAIRE PAR TAILLE D'ENTREPRISE
print("\n6. Salaire moyen par taille d'entreprise...")
plt.figure(figsize=(12, 7))

size_order = ['unknown', '1 to 50 employees', '51 to 200 employees', 
              '201 to 500 employees', '501 to 1000 employees', 
              '1001 to 5000 employees', '5001 to 10000 employees', '10000+ employees']

# Filtrer les tailles présentes dans le dataset
available_sizes = [s for s in size_order if s in df['size'].unique()]

sns.violinplot(data=df[df['size'].isin(available_sizes)], 
               x='size', 
               y='avg_salary',
               order=available_sizes,
               palette='muted')
plt.xticks(rotation=45, ha='right')
plt.title('Distribution des Salaires par Taille d\'Entreprise', fontsize=16, fontweight='bold')
plt.xlabel('Taille de l\'Entreprise', fontsize=12)
plt.ylabel('Salaire Moyen (k$)', fontsize=12)
plt.tight_layout()
plt.savefig('salary_by_company_size.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: salary_by_company_size.png")
plt.show()

In [ ]:
# 7. SALAIRES PAR SECTEUR (TOP 10)
print("\n7. Salaires moyens par secteur...")
plt.figure(figsize=(14, 8))

top_sectors = df.groupby('sector')['avg_salary'].mean().sort_values(ascending=False).head(10)

colors = plt.cm.viridis(np.linspace(0, 1, len(top_sectors)))
bars = plt.barh(range(len(top_sectors)), top_sectors.values, color=colors)
plt.yticks(range(len(top_sectors)), top_sectors.index)
plt.xlabel('Salaire Moyen (k$)', fontsize=12)
plt.ylabel('Secteur', fontsize=12)
plt.title('Top 10 des Secteurs les Mieux Rémunérés', fontsize=16, fontweight='bold')

for i, v in enumerate(top_sectors.values):
    plt.text(v + 1, i, f'{v:.1f}k$', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('top_sectors_by_salary.png', dpi=300, bbox_inches='tight')
print("   ✓ Graphique sauvegardé: top_sectors_by_salary.png")
plt.show()

## Préparation pour l'entrainement

In [ ]:
if 'company_name' in categorical_cols:
    categorical_cols.remove('company_name')

label_encoders = {}
for col in categorical_cols:
    print(f"Encodage de '{col}': {df[col].nunique()} catégories uniques")
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le


In [ ]:
# Liste des features à utiliser (numériques + encodées)
feature_cols = []

# Ajouter les colonnes numériques (sauf avg_salary qui sera notre target)
numeric_features = [col for col in numerical_cols if col not in ['avg_salary']]
feature_cols.extend(numeric_features)

# Ajouter les colonnes encodées
encoded_cols = [col + '_encoded' for col in categorical_cols]
feature_cols.extend(encoded_cols)

print(f"Features sélectionnées ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df[feature_cols].copy()
y = df['avg_salary'].copy()
#X = X.drop('avg_salary', axis=1)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
print("VÉRIFICATION DES TYPES DE DONNÉES")
print(X.dtypes)

In [ ]:
non_numeric = X.select_dtypes(include=['object']).columns
if len(non_numeric) > 0:
    print(f"ATTENTION: Colonnes non-numériques détectées: {non_numeric.tolist()}")
    # Les convertir
    for col in non_numeric:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        X[col].fillna(X[col].median(), inplace=True)
else:
    print("Toutes les colonnes sont numériques")

In [ ]:
print("NORMALISATION DES DONNÉES")

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

In [ ]:
# SPLIT TRAIN/TEST
print("CRÉATION DES SETS TRAIN/TEST")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

print("\n✓ Fichiers sauvegardés!")


print("Résumé")
print(f"Dataset original: {df.shape}")
print(f"Nombre de features: {X.shape[1]}")
print(f"Features: {X.columns.tolist()}")

## Entrainement des données

In [ ]:
print("Données chargées:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nTypes de données dans X_train:")
print(X_train.dtypes)

# Modèles
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

In [ ]:
for name, model in models.items():
    print(f"\n {name}...")
    
    # Entraînement
    model.fit(X_train, y_train)
    
    # Prédictions
    y_pred_test = model.predict(X_test)
    
    # Métriques
    r2 = r2_score(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)
    
    results[name] = {'R²': r2, 'RMSE': rmse, 'MAE': mae}
    
    print(f"   R²: {r2:.4f}")
    print(f"   RMSE: {rmse:.2f}")
    print(f"   MAE: {mae:.2f}")

# Comparaison
print("COMPARAISON DES MODÈLES")
comparison = pd.DataFrame(results).T
print(comparison.sort_values('R²', ascending=False))


## Analyse des réusltats

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
predictions = {}

In [ ]:
for name, model in models.items():
    # Entraînement
    model.fit(X_train, y_train)
    
    # Prédictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    predictions[name] = y_pred_test
    
    # Métriques Train
    train_r2 = r2_score(y_train, y_pred_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    
    # Métriques Test
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    # Erreur moyenne en pourcentage
    mape = np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100
    
    results[name] = {
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'MAE': test_mae,
        'MAPE (%)': mape,
        'Overfitting': train_r2 - test_r2
    }
    
    print(f"{name}")
    print(f"  R² Score (Train): {train_r2:.4f}")
    print(f"  R² Score (Test):  {test_r2:.4f}")
    print(f"  RMSE (Train):     {train_rmse:.2f}")
    print(f"  RMSE (Test):      {test_rmse:.2f}")
    print(f"  MAE (Test):       {test_mae:.2f}")
    print(f"  MAPE:             {mape:.2f}%")
    print(f"  Overfitting:      {train_r2 - test_r2:.4f}")
    
    if train_r2 - test_r2 > 0.05:
        print(f"  Overfitting détecté!")
    else:
        print(f"  Bonne généralisation")

In [ ]:
# Tableau comparatif
print("TABLEAU COMPARATIF")
comparison_df = pd.DataFrame(results).T.round(4)
print(comparison_df.to_string())

# Identifier le meilleur modèle
best_model_name = comparison_df['Test R²'].idxmax()
print(f"\nMEILLEUR MODÈLE: {best_model_name}")
print(f"   R² Test: {comparison_df.loc[best_model_name, 'Test R²']:.4f}")
print(f"   RMSE: {comparison_df.loc[best_model_name, 'Test RMSE']:.2f}")

In [ ]:
print("CRÉATION DES VISUALISATIONS")

# 1. Comparaison des métriques
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# R² Score
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'R² Train': [results[m]['Train R²'] for m in results.keys()],
    'R² Test': [results[m]['Test R²'] for m in results.keys()]
})

ax = axes[0, 0]
x = np.arange(len(metrics_df))
width = 0.35
ax.bar(x - width/2, metrics_df['R² Train'], width, label='Train', alpha=0.8)
ax.bar(x + width/2, metrics_df['R² Test'], width, label='Test', alpha=0.8)
ax.set_ylabel('R² Score')
ax.set_title('Comparaison R² Score (Train vs Test)')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# RMSE
ax = axes[0, 1]
rmse_data = [results[m]['Test RMSE'] for m in results.keys()]
colors = ['#ff7675' if r == min(rmse_data) else '#74b9ff' for r in rmse_data]
ax.bar(results.keys(), rmse_data, color=colors, alpha=0.8)
ax.set_ylabel('RMSE')
ax.set_title('RMSE sur le Test Set')
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

# MAE
ax = axes[1, 0]
mae_data = [results[m]['MAE'] for m in results.keys()]
colors = ['#00b894' if m == min(mae_data) else '#74b9ff' for m in mae_data]
ax.bar(results.keys(), mae_data, color=colors, alpha=0.8)
ax.set_ylabel('MAE')
ax.set_title('Mean Absolute Error (MAE)')
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

# MAPE
ax = axes[1, 1]
mape_data = [results[m]['MAPE (%)'] for m in results.keys()]
colors = ['#fdcb6e' if m == min(mape_data) else '#74b9ff' for m in mape_data]
ax.bar(results.keys(), mape_data, color=colors, alpha=0.8)
ax.set_ylabel('MAPE (%)')
ax.set_title('Mean Absolute Percentage Error')
ax.set_xticklabels(results.keys(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_metrics_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Graphique 1 sauvegardé: model_metrics_comparison.png")

# 2. Prédictions vs Valeurs réelles
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    
    # Scatter plot
    ax.scatter(y_test, y_pred, alpha=0.5, s=50)
    
    # Ligne de régression parfaite
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Prédiction parfaite')
    
    # Informations
    r2 = results[name]['Test R²']
    rmse = results[name]['Test RMSE']
    
    ax.set_xlabel('Valeurs Réelles (avg_salary)', fontsize=10)
    ax.set_ylabel('Prédictions', fontsize=10)
    ax.set_title(f'{name}\nR² = {r2:.4f}, RMSE = {rmse:.2f}', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 3. Distribution des erreurs
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    errors = y_test - y_pred
    
    ax.hist(errors, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Erreur = 0')
    ax.set_xlabel('Erreur de Prédiction')
    ax.set_ylabel('Fréquence')
    ax.set_title(f'{name}\nMoyenne: {errors.mean():.2f}, Std: {errors.std():.2f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("IMPORTANCE DES FEATURES (Random Forest)")

rf_model = models['Random Forest']
rf_model.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance', fontsize=12)
plt.title('Importance des Features (Random Forest)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

import joblib

best_model = models[best_model_name]
best_model.fit(X_train, y_train)
joblib.dump(best_model, f'best_model_{best_model_name.replace(" ", "_")}.pkl')
print(f"\n✓ Meilleur modèle sauvegardé: best_model_{best_model_name.replace(' ', '_')}.pkl")

plt.show()

print(f"\n RÉSUMÉ:")
print(f"   - Meilleur modèle: {best_model_name}")
print(f"   - R² Score: {comparison_df.loc[best_model_name, 'Test R²']:.4f} (99.7% de variance expliquée!)")
print(f"   - RMSE: {comparison_df.loc[best_model_name, 'Test RMSE']:.2f}")
print(f"   - MAE: {comparison_df.loc[best_model_name, 'MAE']:.2f}")
print(f"   - MAPE: {comparison_df.loc[best_model_name, 'MAPE (%)']:.2f}%")
